# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debugging)
* LLM-assisted evaluation

In [1]:
!pip install langchain langchain-openai langchain-community langchain-huggingface
!pip install sentence-transformers docarray
!pip install ragas datasets

from dotenv import load_dotenv, find_dotenv
import os

_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached instructor-1.15.1-py3-none-any.whl.metadata (12 kB)
  Using cached scikit_network-0.33.5.tar.gz (1.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
Using cached ragas-0.4.3-py3-none-any.whl (466 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
Using cached appdirs-1.4.4-py2.py3-none-any.whl (9.6 kB)
Using cached instructor-1.15.1-py3-none-any.whl (178 kB)
Using cached docstring_parser-0.17.0-py3-none-any.whl (36 kB)
F

  error: subprocess-exited-with-error
  
  × Building wheel for scikit-network (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [331 lines of output]
      Compiling ./sknetwork\classification\vote.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\classification\vote.pyx
      Compiling ./sknetwork\clustering\leiden_core.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\clustering\leiden_core.pyx
      Compiling ./sknetwork\clustering\louvain_core.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\clustering\louvain_core.pyx
      Compiling ./sknetwork\hierarchy\paris.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\hierarchy\paris.pyx
      Compiling ./sknetwork\linalg\diteration.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\linalg\diteration.pyx
      Compiling ./sknetwork\linalg\push.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\linalg\push.pyx
      Compiling ./sknetwork\ranking\betweenness.pyx bec

### Example 1

#### Create our QandA application

In [2]:
# Modern imports — langchain_community replaces langchain.document_loaders and langchain.vectorstores
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

# LCEL building blocks (replace LLMChain and RetrievalQA)
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

c:\Users\jesur\Desktop\ACurso\w8\v.w8\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\jesur\Desktop\ACurso\w8\v.w8\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
file = 'data//OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding='utf-8')
data = loader.load()

In [4]:
# Build the vector store directly (replaces VectorstoreIndexCreator)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}
)

vectorstore = DocArrayInMemorySearch.from_documents(data, embeddings)
retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15662.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Modern LCEL RAG chain — replaces RetrievalQA.from_chain_type()
llm = ChatOpenAI(temperature=0.0)

def format_docs(docs):
    """Join retrieved documents with the same separator used in the original notebook."""
    return "<<<<>>>>>".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Use the following pieces of context to answer the user's question.\n"
     "If you don't know the answer, just say that you don't know, "
     "don't try to make up an answer.\n\nContext:\n{context}"),
    ("human", "{question}")
])

qa = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

#### Coming up with test datapoints

In [6]:
data[10]

Document(metadata={'source': 'data//OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\r\n\r\nSize & Fit\r\n- Pants are Favorite Fit: Sits lower on the waist.\r\n- Relaxed Fit: Our most generous fit sits farthest from the body.\r\n\r\nFabric & Care\r\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\r\n\r\nAdditional Features\r\n- Relaxed fit top with raglan sleeves and rounded hem.\r\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\r\n\r\nImported.")

In [7]:
data[11]

Document(metadata={'source': 'data//OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [8]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import BaseOutputParser

# Hard-coded ground-truth examples
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Prompt template for few-shot Q&A
few_shot_prompt = PromptTemplate(
    input_variables=["query"],
    template=(
        "Examples:\n"
        "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
        "   Answer: Yes\n"
        "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
        "   Answer: The DownTek collection\n"
        "Query: {query}\n"
        "Answer:"
    )
)

# Output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Output parser — replaces the old LLMChain output_parser kwarg
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# LCEL chain — replaces LLMChain(llm=..., prompt=..., output_parser=...)
few_shot_chain = few_shot_prompt | llm | AnswerOutputParser()

# Test it
query = "Is the Cozy Comfort Pullover Set available in different colors?"
result = few_shot_chain.invoke({"query": query})
print(result)

answer='Yes, it is available in multiple colors such as grey, navy, and black.'


#### LLM-Generated examples

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(temperature=0)

qa_generate_prompt = PromptTemplate.from_template(
    """Given the following document, generate one good question that can be answered 
    based ONLY on the information in the document.

Document: {doc}

Question:"""
)

example_gen_chain = (
    {"doc": RunnablePassthrough()}
    | qa_generate_prompt
    | llm
    | StrOutputParser()
)

In [10]:
# .batch() replaces the deprecated .apply_and_parse()
new_examples = example_gen_chain.batch(
    [{"doc": t} for t in data[:5]]
)

In [11]:
new_examples[0]

"What is the approximate weight of one pair of Women's Campside Oxfords?"

In [12]:
data[0]

Document(metadata={'source': 'data//OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \r\n\r\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \r\n\r\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \r\n\r\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \r\n\r\nQuestions? Please contact us for any inquiries.")

In [13]:
# Flatten the generated qa_pairs

# Use .batch() instead of .apply()
new_examples = example_gen_chain.batch([{"doc": d.page_content} for d in data[:5]])

#### Combine examples

In [14]:
# Convert the list of generated questions into the dictionary format needed for evaluation
d_flattened = [{"query": q, "answer": "Ground truth not provided"} for q in new_examples]

# Now you can safely append them
examples += d_flattened

In [15]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
 'answer': 'Yes'}

In [16]:
# Test the LCEL chain with the first example
qa.invoke(examples[0]["query"])

'Yes, the Cozy Comfort Pullover Set does have side pockets.'

### Manual Evaluation - Fun part

In [17]:
import langchain
langchain.debug = True

In [18]:
qa.invoke(examples[0]["query"])

'Yes, the Cozy Comfort Pullover Set does have side pockets.'

In [19]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [20]:
# Run all examples through the RAG chain and build predictions list
# The LCEL chain returns a plain string, so we wrap results in the
# {query, answer, result} dict that QAEvalChain.evaluate() expects.
raw_results = qa.batch([ex["query"] for ex in examples])

predictions = [
    {"query": ex["query"], "answer": ex["answer"], "result": raw_results[i]}
    for i, ex in enumerate(examples)
]

predictions

[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes',
  'result': 'Yes, the Cozy Comfort Pullover Set does have side pockets.'},
 {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection',
  'result': 'The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.'},
 {'query': "What is the approximate weight of one pair of Women's Campside Oxfords?",
  'answer': 'Ground truth not provided',
  'result': "The approximate weight of one pair of Women's Campside Oxfords is 1 lb. 1 oz."},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'Ground truth not provided',
  'result': 'The dimensions of the small size of the Recycled Waterhog Dog Mat, Chevron Weave are 18" x 28", and the dimensions of the medium size are 22.5" x 34.5".'},
 {'query': "What features does the Infant and Toddler Girls' Coastal Chil

In [ ]:
!pip install --upgrade langchain langchain-community

In [ ]:
from langchain.evaluation import load_evaluator

# To replace the line that failed in Cell 26 of your notebook:
evaluator = load_evaluator("qa", llm=llm)

ModuleNotFoundError: No module named 'langchain.evaluation'

In [ ]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [ ]:
graded_outputs

In [32]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question:         " + predictions[i]['query'])
    print("Real Answer:      " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Grade:            " + graded_outputs[i].get('results', 'N/A'))
    print()

Example 0:
Question:         Do the Cozy Comfort Pullover Set have side pockets?
Real Answer:      Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set does have side pockets.


NameError: name 'graded_outputs' is not defined

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [30]:
# Build a second RAG pipeline over a plain-text file
loader2 = TextLoader(".//data//nyc_text.txt", encoding='utf-8')
data2 = loader2.load()

embeddings2 = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}   # change to 'mps' on Apple Silicon
)

vectorstore2 = DocArrayInMemorySearch.from_documents(data2, embeddings2)
retriever2 = vectorstore2.as_retriever()

llm2 = ChatOpenAI(temperature=0)

# LCEL chain that also returns source documents
def build_qa_with_sources(retriever, llm):
    """Returns a runnable that outputs {answer, source_documents}."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Use the following context to answer the question.\n"
         "If you don't know the answer, say so.\n\nContext:\n{context}"),
        ("human", "{question}")
    ])

    def retrieve_and_format(question):
        docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)
        return {"docs": docs, "context": context, "question": question}

    from langchain_core.runnables import RunnableLambda

    chain = (
        RunnableLambda(retrieve_and_format)
        | {
            "answer": (lambda x: {"context": x["context"], "question": x["question"]})
                       | prompt | llm | StrOutputParser(),
            "source_documents": lambda x: x["docs"]
          }
    )
    return chain

qa_chain = build_qa_with_sources(retriever2, llm2)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15731.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
# Quick test
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the document
loader2 = TextLoader(".//data//nyc_text.txt", encoding='utf-8')
raw_data2 = loader2.load()

# 2. Split the document into smaller chunks (e.g., 1000 characters each)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)
data2 = text_splitter.split_documents(raw_data2)

# 3. Now build the vector store with the chunks
embeddings2 = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

vectorstore2 = DocArrayInMemorySearch.from_documents(data2, embeddings2)
retriever2 = vectorstore2.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12934.14it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
result

Now in order to evaluate the QA system we generate a few relevant questions. Feel free to add more.

In [25]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

eval_examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

eval_examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### RAGAS Evaluation (modern API — ragas >= 0.2)

The old `EvaluatorChain` wrapper has been removed from ragas. The current API uses `evaluate()` with a Hugging Face `Dataset` object.

Available metrics: `faithfulness`, `answer_relevancy`, `context_recall`, `context_precision`.

In [ ]:
# Collect answers + contexts for all eval questions
result = qa_chain.invoke(eval_questions[1])
result["answer"]

In [ ]:
# Build the full dataset for RAGAS
ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": [],
}

for ex in eval_examples:
    chain_result = qa_chain.invoke(ex["query"])
    ragas_data["question"].append(ex["query"])
    ragas_data["answer"].append(chain_result["answer"])
    ragas_data["contexts"].append(
        [doc.page_content for doc in chain_result["source_documents"]]
    )
    ragas_data["ground_truth"].append(ex["ground_truths"][0])

ragas_data

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

ragas_dataset = Dataset.from_dict(ragas_data)
ragas_dataset

**Faithfulness**

Measures whether the generated answer is grounded in the retrieved context. High score = the answer only contains facts present in the source documents.

In [ ]:
faithfulness_result = evaluate(
    ragas_dataset,
    metrics=[faithfulness]
)
faithfulness_result

High `faithfulness` means there is strong consistency between the source documents and the answer.

You can test lower scores by passing an unrelated answer to the dataset.

In [ ]:
# Inject a fake answer to verify the metric drops
import copy

fake_data = copy.deepcopy(ragas_data)
fake_data["answer"][1] = "we are the champions"

fake_dataset = Dataset.from_dict(fake_data)
evaluate(fake_dataset, metrics=[faithfulness])

**Context Recall**

Measures how much of the ground-truth answer is covered by the retrieved context. High score = the relevant information was successfully retrieved.

In [ ]:
context_recall_result = evaluate(
    ragas_dataset,
    metrics=[context_recall]
)
context_recall_result

High `context_recall` means the ground truth is well represented by the retrieved documents.

Test a lower score by injecting unrelated context:

In [ ]:
fake_data2 = copy.deepcopy(ragas_data)
fake_data2["contexts"][1] = ["I love christmas"]

fake_dataset2 = Dataset.from_dict(fake_data2)
evaluate(fake_dataset2, metrics=[context_recall])

**Full evaluation — all metrics at once**

In [ ]:
full_result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision]
)

# Convert to a pandas DataFrame for easy inspection
full_result.to_pandas()